In [0]:
%run ./transform_data

cette table doit contenir les informations suivantes : 
- alertes
- id_alerting

In [0]:
alerting_ref = {
    1: "This recommendation was not generated due to a tag drop impacted one or more missing automatic values",
    2: "Recommendation not generated due to one or more automatic values being calculated late",
    3: "Recommendation not generated due to an overlap of less than one hour between consecutive localizations",
    4: "This recommendation was not generated in PG despite no missing value",
    5: "This recommendation was not generated in PG because the batch status is in process error",
    6: "This recommendation was not generated in PG because the batch status is pending",
    7: "Batch created too late: recommendation not calculated",
    8: "This recommendation was not generated in PG, not found in the inference monitoring table but expected in batch planning",
    9: "Manual value missing in the DS table despite being entered on time",
    10: "Manual value missing in the DS table because it was entered within 30 minutes before the interval end date",
    11: "Automatic value missing in the DS table but present in measurement_pivot as of job execution : synchronization issue between flows."
}

alerting_data = [(k, v) for k, v in alerting_ref.items()]


schema = StructType([
    StructField("id_alerting", IntegerType(), False),
    StructField("alerte", StringType(), False)
])


dim_alerting = spark.createDataFrame(alerting_data, schema)


In [0]:
dim_alerting_with_ms_idf_cat = dim_alerting.withColumn(
    "ms_idf_cat",
    F.when(F.col("id_alerting") == 1, "MS") #chute tags
    .when(F.col("id_alerting") == 2, "IDF") #mesure en retard
    .when(F.col("id_alerting") == 3, "IDF") #overlap < 1h
    .when(F.col("id_alerting") == 4, "IDF") #no value in PG
    .when(F.col("id_alerting") == 5, "IDF") #status in process error
    .when(F.col("id_alerting") == 6, "IDF") #status pending
    .when(F.col("id_alerting") == 7, "MS") #batch created too late
    .when(F.col("id_alerting") == 8, "IDF") #not found in inference monitoring
    .when(F.col("id_alerting") == 9, "IDF") #missing value in DS
    .when(F.col("id_alerting") == 10, "MS") #missing value in DS
    .when(F.col("id_alerting") == 11, "MS") #missing value in DS
    .otherwise(F.lit(None))
)                                      

Ingestion des données dans la table cible

In [0]:
current_process= "dim_alerting"

In [0]:
target_dim_alerting = current_catalog +"."+current_schema+"."+current_process
print(target_dim_alerting)

In [0]:
all_columns =  dim_alerting_with_ms_idf_cat.columns
display(all_columns)

In [0]:

# define the primary key 
primary_key = [    
    'id_alerting']

additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug': 
    print(additional_columns)

In [0]:
handle_table_update(
    dim_alerting_with_ms_idf_cat, 
    target_dim_alerting, 
    primary_key, 
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode # Use "update" for update mode, "full" for delete/insert mode
    )